# 01 — Exploración y preparación inicial de BDD100K

Este notebook prepara el entorno en Google Colab, descarga los paquetes de imágenes de 10K y las anotaciones de segmentación semántica y por instancias, comprueba la estructura y genera un informe exploratorio. **No descarga los videos de 1.8 TB ni entrena modelos.**

> BDD100K requiere aceptar sus condiciones oficiales de uso. Consulta la [documentación de descarga](https://github.com/bdd100k/bdd100k/blob/master/doc/source/download.rst). Los archivos se guardan en Google Drive para persistir entre sesiones. La descarga puede tardar y depende de la disponibilidad de Colab/red.

## 1. Conectar Google Drive
Ejecuta esta celda y autoriza el acceso a tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/Imagenes_Tema2')
DOWNLOAD_DIR = PROJECT_DIR / 'downloads'
DATA_DIR = PROJECT_DIR / 'data' / 'bdd100k'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Proyecto:', PROJECT_DIR)


## 2. Descargar los paquetes necesarios
Se descargan los tres splits de imágenes 10K y las etiquetas semánticas y de instancias. Las imágenes 10K son el conjunto indicado por BDD100K para estas dos tareas. No descargar `100k_images_*` para este primer flujo: carriles y área transitable usan anotaciones asociadas al conjunto de 100K, no al split 10K.

In [ ]:
import subprocess, os, hashlib, zipfile, shutil, time
BASE_URL = 'https://dl.cv.ethz.ch/bdd100k/data/'
# BDD100K publica un único paquete de imágenes 10K, no ZIP separados por split.
FILES = [
    'bdd100k_images_10k.zip',
    'bdd100k_sem_seg_labels_trainval.zip',
    'bdd100k_ins_seg_labels_trainval.zip',
]
def download_file(filename, retries=5):
    target = DOWNLOAD_DIR / filename
    url = BASE_URL + filename
    if target.exists() and target.stat().st_size > 0:
        print(f'Ya existe: {filename} ({target.stat().st_size/1024**2:.1f} MB)')
        return target
    target.unlink(missing_ok=True)
    for attempt in range(1, retries + 1):
        print(f'Descargando {filename} (intento {attempt}/{retries})')
        result = subprocess.run(
            ['wget', '--https-only', '--continue', '--tries=3', '--timeout=30', '--read-timeout=30',
             '--show-progress', url, '-O', str(target)],
            text=True
        )
        if result.returncode == 0 and target.exists() and target.stat().st_size > 0:
            print(f'Listo: {target.stat().st_size/1024**2:.1f} MB')
            return target
        print(f'wget terminó con código {result.returncode}; se reintentará.')
        target.unlink(missing_ok=True)
        time.sleep(5 * attempt)
    raise RuntimeError(f'No se pudo descargar {filename}. Comprueba la conexión de Colab o inténtalo más tarde: {url}')
for filename in FILES:
    download_file(filename)


## 3. Verificar checksums de las etiquetas
Los MD5 siguientes corresponden a los paquetes de etiquetas indicados por la documentación oficial. No se fuerza una suma para los ZIP de imágenes porque la documentación consultada no publica aquí un MD5 individual por split.

In [ ]:
EXPECTED_MD5 = {
    'bdd100k_images_10k.zip': '08f26aecceda982568063d3d5873378e',
    'bdd100k_sem_seg_labels_trainval.zip': '9a2968dde3345eeb689cffb1e26f9c78',
    'bdd100k_ins_seg_labels_trainval.zip': '651b41f229d7327d8c4af97772de4390',
}
def md5_file(path, chunk_size=8*1024*1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()
for filename, expected in EXPECTED_MD5.items():
    actual = md5_file(DOWNLOAD_DIR / filename)
    print(filename, 'OK' if actual == expected else 'NO COINCIDE', actual)
    if actual != expected:
        raise ValueError(f'Checksum incorrecto para {filename}; elimina el ZIP y vuelve a descargarlo.')


## 4. Extraer los ZIP
La extracción se hace en Drive para conservar los datos al cerrar la sesión. Puede requerir espacio libre considerable.

In [ ]:
def safe_extract(zip_path, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        root = destination.resolve()
        for member in zf.infolist():
            resolved = (destination / member.filename).resolve()
            if root not in resolved.parents and resolved != root:
                raise RuntimeError(f'Ruta insegura en ZIP: {member.filename}')
        zf.extractall(destination)
    print('Extraído:', Path(zip_path).name)
for filename in FILES:
    safe_extract(DOWNLOAD_DIR / filename, DATA_DIR)
print('Extracción terminada en', DATA_DIR)


## 5. Inspeccionar carpetas, conteos y archivos de anotación

In [ ]:
for p in sorted(DATA_DIR.rglob('*')):
    if p.is_dir() and any(token in str(p).lower() for token in ['images/10k','sem_seg/masks','ins_seg/bitmasks']):
        files = [x for x in p.iterdir() if x.is_file()]
        print(f'{p}: {len(files)} archivos directos')

image_root = DATA_DIR / 'bdd100k' / 'images' / '10k'
if not image_root.exists():
    candidates = list(DATA_DIR.rglob('10k'))
    print('No se encontró la ruta esperada. Carpetas candidatas:', candidates[:10])
else:
    for split in ['train','val','test']:
        folder = image_root / split
        imgs = list(folder.glob('*')) if folder.exists() else []
        print(split, '->', len(imgs), 'archivos')

print('\nJSON de anotaciones:')
for p in DATA_DIR.rglob('*.json'):
    print(p.relative_to(DATA_DIR), f'({p.stat().st_size/1024**2:.1f} MB)')


## 6. Visualizar una imagen y su máscara semántica (si están presentes)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
img_dir = DATA_DIR / 'bdd100k' / 'images' / '10k' / 'train'
mask_dir = DATA_DIR / 'bdd100k' / 'labels' / 'sem_seg' / 'masks' / 'train'
if img_dir.exists() and mask_dir.exists():
    imgs = sorted(img_dir.glob('*.jpg'))
    if imgs:
        img_path = imgs[0]
        mask_path = mask_dir / (img_path.stem + '.png')
        fig, ax = plt.subplots(1, 2 if mask_path.exists() else 1, figsize=(14,5))
        ax0 = ax[0] if isinstance(ax, (list, tuple)) or hasattr(ax, '__len__') else ax
        ax0.imshow(Image.open(img_path)); ax0.set_title(img_path.name); ax0.axis('off')
        if mask_path.exists():
            ax[1].imshow(Image.open(mask_path)); ax[1].set_title('Máscara semántica'); ax[1].axis('off')
        plt.show()
    else:
        print('No se encontraron JPG en', img_dir)
else:
    print('No se localizaron las rutas esperadas; revisa la estructura impresa en la sección 5.')


## 7. Metadatos ambientales: alcance y limitación
BDD100K documenta atributos `weather`, `scene` y `timeofday` en los JSON de anotación de detección asociados a las imágenes/vídeos de 100K. Sin embargo, las imágenes 10K usadas para segmentación semántica e instancias **no constituyen un subconjunto exacto** de las 100K. Por tanto, no se debe asignar una condición ambiental por suposición. En una siguiente etapa habrá que obtener los JSON de metadatos, cruzar por nombre/ID de imagen y reportar la cobertura real; las imágenes sin correspondencia quedarán como `unknown`.

## 8. Guardar un informe de exploración

In [ ]:
from datetime import datetime
report = [
    '# Informe inicial BDD100K',
    f'Fecha de ejecución: {datetime.now().isoformat(timespec="seconds")}',
    f'Ruta de datos: `{DATA_DIR}`',
    '', '## Archivos descargados',
]
for filename in FILES:
    p = DOWNLOAD_DIR / filename
    report.append(f'- {filename}: {p.stat().st_size/1024**3:.2f} GiB' if p.exists() else f'- {filename}: FALTA')
report += ['', '## Conteo de imágenes']
for split in ['train','val','test']:
    folder = image_root / split
    count = len(list(folder.glob('*.jpg'))) if folder.exists() else 0
    report.append(f'- {split}: {count}')
report += ['', '## Próximos pasos', '- Verificar correspondencia imagen-máscara.', '- Crear el mapeo de clases y los dataloaders.', '- Cruzar metadatos ambientales con los IDs que tengan correspondencia.', '- Entrenar primero con un subconjunto pequeño en Colab antes de escalar.']
report_path = PROJECT_DIR / 'informe_exploracion_bdd100k.md'
report_path.write_text('\n'.join(report), encoding='utf-8')
print(report_path.read_text())


## Notas para el siguiente notebook
- Para BDD100K, semantic segmentation e instance segmentation usan los datos de 10K y sus respectivos paquetes de etiquetas.
- Lane marking y drivable area se anotan sobre 100K Images; no deben mezclarse como si fueran máscaras de los 10K.
- El entrenamiento completo de dos arquitecturas puede exceder el tiempo/GPU disponible en Colab gratuito. Empezar con un subconjunto, resolución moderada, batch pequeño y checkpoints en Drive.
- No subir los datos ni los ZIP al repositorio GitHub.